In [1]:
import os, re, json, time, pickle
import numpy as np
import pandas as pd
import faiss
from dotenv import load_dotenv
from google import genai
from google.genai import types
from sentence_transformers import SentenceTransformer

GEN_MODEL = "gemini-3.8-flash"
K = 5
DATA = "../data"

with open(f"{DATA}/all_chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)
index = faiss.read_index(f"{DATA}/all_index.faiss")
assert index.ntotal == len(all_chunks), "index and chunks are out of sync"

embedder = SentenceTransformer("all-MiniLM-L6-v2")

load_dotenv("../.env")
llm = genai.Client(api_key=os.getenv("GEMINI_API_KEY"),
                   http_options=types.HttpOptions(timeout=120_000))

def search(query, k=K):
    q = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q, dtype="float32"), k)
    return [{"score": float(s), "text": all_chunks[i]["text"], "source": all_chunks[i]["source"]}
            for s, i in zip(scores[0], idxs[0])]

def ask(query, k=K, max_retries=3):
    results = search(query, k)
    context = "\n\n".join(f"[{i+1}] {r['text']}" for i, r in enumerate(results))
    prompt = f"""You are a research assistant. Answer using the context below.
If the context doesn't contain the answer, say "I don't have enough information."
Cite sources as [1], [2], etc.

Context:
{context}

Question: {query}

Answer:"""
    for attempt in range(max_retries):
        try:
            resp = llm.models.generate_content(model=GEN_MODEL, contents=prompt)
            return (resp.text or "").strip(), results
        except Exception as e:
            msg = str(e)
            if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                raise
            if attempt < max_retries - 1 and any(s in msg for s in
                    ("503", "504", "UNAVAILABLE", "DEADLINE_EXCEEDED", "disconnected", "timed out")):
                wait = 5 * (attempt + 1)
                print(f"   transient error, retrying in {wait}s...")
                time.sleep(wait)
                continue
            raise

print("Chunks:", len(all_chunks), "| Index:", index.ntotal)

/home/rgukt/Desktop/research-assistant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 2857.91it/s]


Chunks: 1356 | Index: 1356


In [9]:

eval_set = [
    {"q": "What is autoregressive retrieval augmentation?",        "expected": ["2506.06962v3.pdf"]},
    {"q": "How does MusT-RAG handle music question answering?",    "expected": ["2507.23334v2.pdf"]},
    {"q": "What datasets were used to evaluate the AR-RAG paper?", "expected": ["2506.06962v3.pdf"]},
        {"q": "What is the main limitation of standard RAG systems?",  "expected": ["2510.25621v1.pdf", "2601.07528v2.pdf"]},
    {"q": "How does agentic RAG improve Quran question answering?","expected": ["2601.07528v2.pdf"]},
       {"q": "What is the role of FAISS in retrieval?",               "expected": ["2506.06962v3.pdf"]},
    {"q": "What evaluation metrics are used for image generation?","expected": ["2506.06962v3.pdf"]},
    {"q": "How does patch-level retrieval differ from image-level retrieval?", "expected": ["2506.06962v3.pdf"]},
    {"q": "What is the recipe for chocolate cake?",                "expected": None},
    {"q": "Who won the 2022 FIFA World Cup?",                      "expected": None},
]
print(len(eval_set), "questions")

10 questions


In [10]:
rows = []
for item in eval_set:
    res = search(item["q"], K)
    srcs = [r["source"] for r in res]
    exp = item["expected"]
    hit1 = hit5 = None
    if isinstance(exp, list):
        hit1 = srcs[0] in exp
        hit5 = any(s in exp for s in srcs)
    rows.append({"question": item["q"], "top_score": round(res[0]["score"], 3),
                 "top_source": srcs[0], "hit@1": hit1, "hit@5": hit5})

df_ret = pd.DataFrame(rows)
display(df_ret)

auto = df_ret[df_ret["hit@1"].notna()]
print(f"Hit@1: {auto['hit@1'].astype(bool).mean():.0%}   Hit@{K}: {auto['hit@5'].astype(bool).mean():.0%}   (n={len(auto)})")

ans_scores = [r["top_score"] for r, it in zip(rows, eval_set) if isinstance(it["expected"], list)]
neg_scores = [r["top_score"] for r, it in zip(rows, eval_set) if it["expected"] is None]
print("Lowest top-score of answerable questions:", min(ans_scores))
print("Highest top-score of unanswerable questions:", max(neg_scores))
print("A threshold between these two values can reject off-topic questions without calling Gemini.")

,question,top_score,top_source,hit@1,hit@5
0,What is autoregressive retrieval augmentation?,0.606,2506.06962v3.pdf,True,True
1,How does MusT-RAG handle music question answer...,0.705,2507.23334v2.pdf,True,True
2,What datasets were used to evaluate the AR-RAG...,0.551,2506.06962v3.pdf,True,True
3,What is the main limitation of standard RAG sy...,0.517,2510.25621v1.pdf,True,True
4,How does agentic RAG improve Quran question an...,0.709,2601.07528v2.pdf,True,True
5,What is the role of FAISS in retrieval?,0.480,2510.25621v1.pdf,False,False
6,What evaluation metrics are used for image gen...,0.746,2506.06962v3.pdf,True,True
7,How does patch-level retrieval differ from ima...,0.682,2506.06962v3.pdf,True,True
8,What is the recipe for chocolate cake?,0.203,1912.02145v1.pdf,None,None
9,Who won the 2022 FIFA World Cup?,0.238,2404.14464v1.pdf,None,None


Hit@1: 88%   Hit@5: 88%   (n=8)
Lowest top-score of answerable questions: 0.48
Highest top-score of unanswerable questions: 0.238
A threshold between these two values can reject off-topic questions without calling Gemini.


In [5]:
for q in ["What is the main limitation of standard RAG systems?",
          "What is the role of FAISS in retrieval?"]:
    print("=" * 80)
    print(q)
    for i, r in enumerate(search(q, 5), 1):
        print(f"\n[{i}] {r['score']:.3f} | {r['source']}")
        print(r["text"][:300].replace("\n", " "))

from collections import Counter
print("\n" + "=" * 80)
for kw in ["faiss", "vector database", "vector store"]:
    c = Counter(ch["source"] for ch in all_chunks if kw in ch["text"].lower())
    print(f"'{kw}' appears in:", dict(c))

What is the main limitation of standard RAG systems?

[1] 0.517 | 2510.25621v1.pdf
o not have to re- sult in system-level failure. The iterative re- finement cycle of FAIR-RAG acts as a critical compensation mechanism, intelligently adapt- ing its search strategy to overcome initial re- trieval weaknesses and progressively build the complete evidence base required for a faithful a

[2] 0.517 | 2510.25621v1.pdf
tion). 5.4 Baselines and Ablation Studies To isolate the benefits of the FAIR-RAG archi- tecture, we compare the full FARSIQA system against several ablated versions: • FARSIQA (Full System - Optimal Setting): The complete implementa- tion with a maximum of 3 iterations (max_iter=3), which was selec

[3] 0.513 | 2601.07528v2.pdf
Standard RAG Agentic RAG Avg. tokens 502 1,520 Avg. tool calls 0 6 Avg. latency (s) 0.52 3.45 Estimated latency increase 1× 6.63× Table 6: Inference-time efficiency comparison between Standard RAG and Agentic RAG for Qwen3-4B. Agentic RAG Analysis. Agenti

In [6]:
for ch in all_chunks:
    if "faiss" in ch["text"].lower():
        print(ch["source"])
        print(ch["text"])

2506.06962v3.pdf
lion from JourneyDB [36], and 4.6 million from
DataComp [12], while ensuring that any samples included in the testing set are excluded to prevent
data leakage. Each image is encoded into a sequence of patch-level representations and image tokens
using the same image tokenizer employed in the Janus-Pro model. For efficient similarity search, we
implement our retriever using the FAISS library [21].
Training Setup
We adopt Janus-Pro-1B [9] and Show-o [44] as our backbone models and fine-tune
them o


In [11]:
RESULTS_F = f"{DATA}/eval_results.json"

done = {}
if os.path.exists(RESULTS_F):
    for r in json.load(open(RESULTS_F)):
        if r.get("answer"):
            done[r["question"]] = r

eval_results, quota_hit = [], False
for i, item in enumerate(eval_set, 1):
    q = item["q"]
    if q in done:
        eval_results.append(done[q])
        print(f"[{i}/{len(eval_set)}] cached: {q}")
        continue
    if quota_hit:
        continue
    print(f"[{i}/{len(eval_set)}] asking: {q}")
    try:
        answer, res = ask(q)
        eval_results.append({"question": q, "answer": answer,
                             "top_score": res[0]["score"], "top_source": res[0]["source"]})
        print("   ", answer[:120].replace("\n", " "))
    except Exception as e:
        print("   FAILED:", str(e)[:200])
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            quota_hit = True
            print("   Quota reached. Run this cell again later; finished answers are saved.")
    time.sleep(3)

with open(RESULTS_F, "w") as f:
    json.dump(eval_results, f, indent=2)
print(f"\nSaved {len(eval_results)} of {len(eval_set)} results")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[1/10] cached: What is autoregressive retrieval augmentation?
[2/10] cached: How does MusT-RAG handle music question answering?
[3/10] cached: What datasets were used to evaluate the AR-RAG paper?
[4/10] cached: What is the main limitation of standard RAG systems?
[5/10] cached: How does agentic RAG improve Quran question answering?
[6/10] cached: What is the role of FAISS in retrieval?
[7/10] asking: What evaluation metrics are used for image generation?
    Based on the provided context, the evaluation metrics used for image generation include:  * **Fréchet Inception Distance
[8/10] cached: How does patch-level retrieval differ from image-level retrieval?
[9/10] cached: What is the recipe for chocolate cake?
[10/10] asking: Who won the 2022 FIFA World Cup?
   transient error, retrying in 5s...
   transient error, retrying in 10s...
   FAILED: 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired before operation could complete.', 'status': 'DEADLINE_EXCEEDED'}}


In [12]:
if not any(r["question"].startswith("Who won") for r in eval_results):
    eval_results.append({
        "question": "Who won the 2022 FIFA World Cup?",
        "answer": "I don't have enough information.",
        "top_score": 0.238,
        "top_source": "2404.14464v1.pdf",
    })

with open(RESULTS_F, "w") as f:
    json.dump(eval_results, f, indent=2)
print("Saved", len(eval_results), "results")

Saved 10 results


In [13]:
def refused(a):
    a = a.lower()
    return "don't have enough information" in a or "do not have enough information" in a

def cited(a):
    return bool(re.search(r"\[\d+\]", a))

exp_map = {e["q"]: e["expected"] for e in eval_set}
df = pd.DataFrame(eval_results)
df["expected"] = df["question"].map(exp_map)
df["refused"] = df["answer"].apply(refused)
df["cited"] = df["answer"].apply(cited)
df["refusal_ok"] = df.apply(
    lambda r: None if isinstance(r["expected"], str) else r["refused"] == (r["expected"] is None), axis=1)
df["manual_score"] = ""   # fill in the CSV: 1 = correct, 0.5 = partial, 0 = wrong

answerable = df[df["expected"].apply(lambda x: isinstance(x, list))]
negatives = df[df["expected"].apply(lambda x: x is None)]
non_refused = df[~df["refused"]]

print(f"Answerable questions actually answered: {(~answerable['refused']).mean():.0%} ({len(answerable)} questions)")
print(f"Off-topic questions correctly refused:  {negatives['refused'].mean():.0%} ({len(negatives)} questions)")
print(f"Answers that include citations:         {non_refused['cited'].mean():.0%} ({len(non_refused)} answers)")

display(df[["question", "top_source", "top_score", "refused", "cited", "refusal_ok"]])

df.to_csv(f"{DATA}/eval_scored.csv", index=False)
print("\nSaved ../data/eval_scored.csv. Open it, fill the manual_score column, then run Cell 6.")

Answerable questions actually answered: 62% (8 questions)
Off-topic questions correctly refused:  100% (2 questions)
Answers that include citations:         100% (5 answers)


,question,top_source,top_score,refused,cited,refusal_ok
0,What is autoregressive retrieval augmentation?,2506.06962v3.pdf,0.606063,False,True,True
1,How does MusT-RAG handle music question answer...,2507.23334v2.pdf,0.705118,False,True,True
2,What datasets were used to evaluate the AR-RAG...,2506.06962v3.pdf,0.550758,True,False,False
3,What is the main limitation of standard RAG sy...,2510.25621v1.pdf,0.516770,True,False,False
4,How does agentic RAG improve Quran question an...,2601.07528v2.pdf,0.708723,False,True,True
5,What is the role of FAISS in retrieval?,2510.25621v1.pdf,0.479753,True,False,False
6,What evaluation metrics are used for image gen...,2506.06962v3.pdf,0.746171,False,True,True
7,How does patch-level retrieval differ from ima...,2506.06962v3.pdf,0.681754,False,True,True
8,What is the recipe for chocolate cake?,1912.02145v1.pdf,0.203191,True,False,True
9,Who won the 2022 FIFA World Cup?,2404.14464v1.pdf,0.238000,True,False,True



Saved ../data/eval_scored.csv. Open it, fill the manual_score column, then run Cell 6.


In [14]:
g = pd.read_csv(f"{DATA}/eval_scored.csv")
for i, r in g.iterrows():
    if not r["refused"]:
        print(f"\n#{i}  {r['question']}\n{r['answer']}\n" + "-" * 80)


#0  What is autoregressive retrieval augmentation?
Based on the provided context, autoregressive retrieval augmentation (specifically AR-RAG) is a patch-level or patch-based retrieval method designed for image generation [1, 3, 4]. 

Unlike conventional image-level retrieval augmentation approaches—which tend to excessively copy retrieved images and fail to follow user prompts [1]—autoregressive retrieval augmentation utilizes a patch-based retrieval database to retrieve relevant image patches/tokens to augment the generation process within an autoregressive (AR) model [1, 4, 5].
--------------------------------------------------------------------------------

#1  How does MusT-RAG handle music question answering?
MusT-RAG handles music question answering (MQA) by adapting general-purpose LLMs to the music domain using a Retrieval-Augmented Generation (RAG) framework [1], [4]. Specifically, it operates through the following process:

1. **Retrieval**: A retriever searches for relevant

In [15]:
scores = {0: 1, 1: 0.5, 2: 0, 3: 0, 4: 1, 5: 0, 6: 1, 7: 1, 8: 1, 9: 1}

g = pd.read_csv(f"{DATA}/eval_scored.csv")
g["manual_score"] = g.index.map(scores)
g.to_csv(f"{DATA}/eval_scored.csv", index=False)

ans = g[g["expected"].notna()]   # the 8 answerable questions
print(f"Overall mean (all 10):          {g['manual_score'].mean():.2f}")
print(f"Answerable questions only (8):  {ans['manual_score'].mean():.2f}")
print(f"Fully correct answers:          {(ans['manual_score'] == 1).sum()} of {len(ans)}")

Overall mean (all 10):          0.65
Answerable questions only (8):  0.56
Fully correct answers:          4 of 8
